# 📊 Exploratory Data Analysis - Property Valuation

This notebook performs EDA on the property dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
# Load data
DATA_DIR = Path('../data/processed')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Training data: {train_df.shape}')
print(f'Test data: {test_df.shape}')

In [ ]:
# Data overview
train_df.head()

In [ ]:
# Data types and missing values
train_df.info()

In [ ]:
# Price statistics
print('Price Statistics:')
print(train_df['price'].describe())

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df['price'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(train_df['price'].mean(), color='red', linestyle='--', label=f'Mean: ${train_df["price"].mean():,.0f}')
axes[0].axvline(train_df['price'].median(), color='green', linestyle='--', label=f'Median: ${train_df["price"].median():,.0f}')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Price Distribution')
axes[0].legend()

axes[1].hist(np.log1p(train_df['price']), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Log(Price + 1)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log-Transformed Price Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_df = train_df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Price correlations
if 'price' in numeric_df.columns:
    correlations = numeric_df.corr()['price'].drop('price').sort_values()
    
    plt.figure(figsize=(10, 8))
    colors = ['green' if x > 0 else 'red' for x in correlations]
    correlations.plot(kind='barh', color=colors)
    plt.xlabel('Correlation with Price')
    plt.title('Feature Correlations with Price')
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# Geographic distribution
if 'lat' in train_df.columns and 'long' in train_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    scatter = axes[0].scatter(train_df['long'], train_df['lat'], 
                               c=train_df['price'], cmap='viridis', alpha=0.5, s=5)
    axes[0].set_xlabel('Longitude')
    axes[0].set_ylabel('Latitude')
    axes[0].set_title('Property Locations (colored by price)')
    plt.colorbar(scatter, ax=axes[0], label='Price ($)')
    
    sns.kdeplot(x=train_df['long'], y=train_df['lat'], ax=axes[1], 
                cmap='Reds', fill=True, levels=20)
    axes[1].set_xlabel('Longitude')
    axes[1].set_ylabel('Latitude')
    axes[1].set_title('Property Density')
    
    plt.tight_layout()
    plt.show()